# Supervised Learning: All Methods on One Dataset

This notebook fits 10 supervised learning methods on the same 2D dataset and visualizes their decision boundaries side-by-side. See how linear and nonlinear methods differ.

**Run All:** Runtime -> Run all (or Ctrl+F9).

Links: [Data Science BSc Course - GitHub](https://github.com/Digital-AI-Finance/data-science) | [Dashboard](https://digital-ai-finance.github.io/data-science/)

## What is a decision boundary?

A decision boundary is the line (or curve) that separates predicted classes. A flat line means a linear method; a curved or wiggly boundary means a nonlinear method. Looking at the shape of the boundary tells you a lot about the algorithm.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, RidgeClassifier, LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
NAVY, AMBER = '#1e3a5f', '#d97706'

X, y = make_moons(n_samples=300, noise=0.25, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)

plt.figure(figsize=(6, 4))
plt.scatter(X[y==0, 0], X[y==0, 1], c=NAVY, s=25, edgecolor='k', linewidth=0.5, label='Class 0')
plt.scatter(X[y==1, 0], X[y==1, 1], c=AMBER, s=25, edgecolor='k', linewidth=0.5, label='Class 1')
plt.title('Two-class problem: make_moons (n=300, noise=0.25)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
class OLSClassifier:
    """OLS regression thresholded at 0.5 (pedagogical anti-pattern)."""
    def fit(self, X, y):
        self.m = LinearRegression().fit(X, y)
        return self
    def predict(self, X):
        return (self.m.predict(X) > 0.5).astype(int)

models = {
    # Row 1: Linear methods
    'OLS (threshold)':     OLSClassifier(),
    'Ridge Classifier':    RidgeClassifier(alpha=100, random_state=42),
    'Lasso (L1 LogReg)':   LogisticRegression(penalty='l1', solver='saga', C=0.01, random_state=42, max_iter=10000),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=5000),
    'Naive Bayes':         GaussianNB(),
    # Row 2: Nonlinear methods
    'Decision Tree':       DecisionTreeClassifier(max_depth=3, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=50, random_state=42),
    'Gradient Boosting':   HistGradientBoostingClassifier(random_state=42),
    'SVM (RBF)':           SVC(kernel='rbf', gamma=2, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=15),
}

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
xx, yy = np.meshgrid(
    np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 300),
    np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 300),
)
grid = np.c_[xx.ravel(), yy.ravel()]

results = []
for ax, (name, model) in zip(axes.ravel(), models.items()):
    t0 = time.time()
    model.fit(X_tr, y_tr)
    t1 = time.time()
    acc = (model.predict(X_te) == y_te).mean()
    Z = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu', levels=[-0.5, 0.5, 1.5])
    ax.scatter(X_tr[y_tr==0, 0], X_tr[y_tr==0, 1], c=NAVY, s=15, edgecolor='k', linewidth=0.5)
    ax.scatter(X_tr[y_tr==1, 0], X_tr[y_tr==1, 1], c=AMBER, s=15, edgecolor='k', linewidth=0.5)
    ax.set_title(f'{name}\nAccuracy: {acc:.2%}', fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
    results.append({'Method': name, 'Accuracy': acc, 'Train time (s)': t1 - t0})

fig.suptitle('Supervised Learning: Decision Boundaries (Row 1 = Linear, Row 2 = Nonlinear)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
df = pd.DataFrame(results).sort_values('Accuracy', ascending=False).reset_index(drop=True)
df['Accuracy'] = df['Accuracy'].apply(lambda x: f'{x:.2%}')
df['Train time (s)'] = df['Train time (s)'].apply(lambda x: f'{x:.4f}')
df

## Your Turn: Parameter Exercise

The accuracies above depend heavily on hyperparameter choices. Let's see how much.

Try these experiments by calling `compare(...)` with different parameters. Each experiment reveals a different lesson:

**Experiment 1 - Regularization convergence:** Set `ridge_alpha=0.01` (almost no regularization). Ridge should now match OLS.
```python
compare(ridge_alpha=0.01)
```

**Experiment 2 - L1 penalty strength:** Set `lasso_C=100` (almost no regularization). Lasso should now match Logistic Regression.
```python
compare(lasso_C=100)
```

**Experiment 3 - KNN extremes:** Try `knn_k=1` (memorizes every point) vs `knn_k=50` (averages 1/6 of the training set).
```python
compare(knn_k=1)
compare(knn_k=50)
```

**Experiment 4 - Tree depth extremes:** Try `tree_depth=1` (a single stump) vs `tree_depth=20` (fully grown).
```python
compare(tree_depth=1)
compare(tree_depth=20)
```

**Experiment 5 - Data difficulty:** Try `noise=0.05` (clean moons) vs `noise=0.45` (heavily overlapping).
```python
compare(noise=0.05)
compare(noise=0.45)
```

Run one at a time, observe how the 2x5 grid changes, and predict what the accuracy table will show BEFORE you look.

In [ ]:
def compare(ridge_alpha=100, lasso_C=0.01, logreg_C=1.0,
            knn_k=15, tree_depth=3, rf_n=50, svm_gamma=2,
            noise=0.25, n_samples=300):
    """Fit all 10 models with custom hyperparameters and visualize decision boundaries.

    Parameters
    ----------
    ridge_alpha : float, default=100
        Ridge regularization strength. Larger = simpler model.
    lasso_C : float, default=0.01
        Inverse L1 regularization for Lasso-style LogReg. Smaller = stronger penalty.
    logreg_C : float, default=1.0
        Inverse L2 regularization for Logistic Regression.
    knn_k : int, default=15
        Number of neighbors for KNN. Larger = smoother boundary.
    tree_depth : int, default=3
        Maximum tree depth. Larger = more complex boundary.
    rf_n : int, default=50
        Number of trees in Random Forest.
    svm_gamma : float, default=2
        RBF kernel bandwidth. Larger = tighter boundary around each point.
    noise : float, default=0.25
        Moon-dataset noise level. Larger = harder separation.
    n_samples : int, default=300
        Total number of samples.

    Returns
    -------
    pandas.DataFrame sorted by accuracy (descending).

    Examples
    --------
    >>> compare(ridge_alpha=0.01)       # Ridge becomes nearly OLS
    >>> compare(lasso_C=100)             # Lasso becomes nearly LogReg
    >>> compare(knn_k=1, tree_depth=20)  # Two ways to overfit
    >>> compare(noise=0.05)              # Clean data: all methods near 100%
    """
    X, y = make_moons(n_samples=n_samples, noise=noise, random_state=42)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)

    local_models = {
        'OLS (threshold)':     OLSClassifier(),
        'Ridge Classifier':    RidgeClassifier(alpha=ridge_alpha, random_state=42),
        'Lasso (L1 LogReg)':   LogisticRegression(penalty='l1', solver='saga', C=lasso_C, random_state=42, max_iter=10000),
        'Logistic Regression': LogisticRegression(C=logreg_C, random_state=42, max_iter=5000),
        'Naive Bayes':         GaussianNB(),
        'Decision Tree':       DecisionTreeClassifier(max_depth=tree_depth, random_state=42),
        'Random Forest':       RandomForestClassifier(n_estimators=rf_n, random_state=42),
        'Gradient Boosting':   HistGradientBoostingClassifier(random_state=42),
        'SVM (RBF)':           SVC(kernel='rbf', gamma=svm_gamma, random_state=42),
        'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=knn_k),
    }

    fig, axes = plt.subplots(2, 5, figsize=(18, 7))
    xx, yy = np.meshgrid(
        np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 200),
        np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 200),
    )
    grid = np.c_[xx.ravel(), yy.ravel()]

    rows = []
    for ax, (name, model) in zip(axes.ravel(), local_models.items()):
        model.fit(X_tr, y_tr)
        acc = (model.predict(X_te) == y_te).mean()
        Z = model.predict(grid).reshape(xx.shape)
        ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu', levels=[-0.5, 0.5, 1.5])
        ax.scatter(X_tr[y_tr==0, 0], X_tr[y_tr==0, 1], c=NAVY, s=15, edgecolor='k', linewidth=0.5)
        ax.scatter(X_tr[y_tr==1, 0], X_tr[y_tr==1, 1], c=AMBER, s=15, edgecolor='k', linewidth=0.5)
        ax.set_title(f'{name}\nAccuracy: {acc:.2%}', fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
        rows.append({'Method': name, 'Accuracy': f'{acc:.2%}'})

    fig.suptitle(f'compare(ridge_alpha={ridge_alpha}, lasso_C={lasso_C}, knn_k={knn_k}, tree_depth={tree_depth}, noise={noise})', fontsize=10)
    plt.tight_layout()
    plt.show()
    return pd.DataFrame(rows).sort_values('Accuracy', ascending=False).reset_index(drop=True)

# Example: show what happens with near-zero Ridge regularization
compare(ridge_alpha=0.01)

## Expected Results

Here is what each experiment should reveal:

| Experiment | Call | What Changes | Lesson |
|---|---|---|---|
| Baseline | (default) | Ridge flat, Lasso ~55%, 40pp spread | Strong regularization hurts linear models on simple 2D data |
| Ridge to OLS | `compare(ridge_alpha=0.01)` | Ridge boundary identical to OLS | Regularization disappears as alpha to 0 |
| Lasso to LogReg | `compare(lasso_C=100)` | Lasso boundary matches Logistic Regression | L1 penalty disappears as C to infinity |
| KNN overfit | `compare(knn_k=1)` | Jagged, noisy boundary | High variance, low bias |
| KNN underfit | `compare(knn_k=50)` | Very smooth boundary, lower accuracy | High bias, low variance |
| Tree stump | `compare(tree_depth=1)` | Single axis-aligned split | Cannot capture the curve |
| Tree deep | `compare(tree_depth=20)` | Jagged boundary fitting every training point | Overfitting |
| Clean data | `compare(noise=0.05)` | Almost all methods reach ~100% | Easy separation |
| Noisy data | `compare(noise=0.45)` | All methods suffer; nonlinear still wins | Bayes error limit |

### Big picture

**Hyperparameter tuning matters as much as algorithm choice.** Your Logistic Regression with the right `C` often beats Random Forest with the wrong `max_depth`. In practice, use `GridSearchCV` or `RandomizedSearchCV` to automate this - but always check by hand first to build intuition.

The next step: use cross-validation to choose the best hyperparameters, not visual inspection.

## Key takeaways

- **Linear methods (row 1)** draw flat or gently-curved boundaries. They fail on crescent-shaped moons data because no straight line can separate the classes cleanly.
- **Nonlinear methods (row 2)** draw curved boundaries that wrap around the moon shape. Accuracy is visibly higher.
- **You can see the accuracy differences BEFORE reading the numbers** - just look at how well the shaded regions match the point colors.
- **Trade-off: complexity vs overfitting.** Decision Tree has jagged edges that fit the training data too closely. Gradient Boosting produces smoother boundaries that generalize better.
- **OLS-as-classifier is a pedagogical anti-pattern** - use Logistic Regression for real classification work.

## Next steps

Go deeper on any single method with the per-lesson notebooks:
- [L21 Linear Regression](https://colab.research.google.com/github/Digital-AI-Finance/data-science-notebooks/blob/main/L21_Linear_Regression/colab/L21_Linear_Regression_Notebook.ipynb)
- [L22 Regularization (Ridge, Lasso, ElasticNet)](https://colab.research.google.com/github/Digital-AI-Finance/data-science-notebooks/blob/main/L22_Regularization/colab/L22_Regularization_Notebook.ipynb)
- [L25 Logistic Regression](https://colab.research.google.com/github/Digital-AI-Finance/data-science-notebooks/blob/main/L25_Logistic_Regression/colab/L25_Logistic_Regression_Notebook.ipynb)
- [L26 Decision Trees](https://colab.research.google.com/github/Digital-AI-Finance/data-science-notebooks/blob/main/L26_Decision_Trees/colab/L26_Decision_Trees_Notebook.ipynb)

Resources:
- [Course Dashboard](https://digital-ai-finance.github.io/data-science/)
- [Supervised Learning Complete Handout](https://digital-ai-finance.github.io/data-science/additional_lectures/Supervised_Learning_Complete.pdf)